# 08 - Snowflake Cortex Agent Tool Preflight

This notebook models a Cortex Agent custom-tool pattern: before the agent answers with data or invokes an operational tool, it calls Metatate through managed MCP and records the decision.

The notebook runs locally for repeatability, but the contract is the same one a Snowflake-hosted custom tool can enforce.


In [ ]:
from pathlib import Path
import json
import os
import sys

import pandas as pd

repo_root = Path.cwd()
if repo_root.name == "notebooks":
    repo_root = repo_root.parent
sys.path.insert(0, str(repo_root))

from common import get_client

client = get_client()
mode = os.getenv("METATATE_EXAMPLES_MODE", "offline")
print(f"Metatate examples mode: {mode}")


def decision_label(response):
    data = response.get("data", {})
    decision = data.get("decision")
    if isinstance(decision, dict):
        return decision.get("decision") or decision.get("overall_decision") or data.get("overall_decision", "UNKNOWN")
    return decision or data.get("overall_decision", "UNKNOWN")


def rationale_text(response):
    data = response.get("data", {})
    decision = data.get("decision")
    if isinstance(decision, dict):
        return decision.get("rationale") or data.get("summary") or ""
    return data.get("rationale") or data.get("summary") or ""


def agent_action_text(response):
    action = response.get("data", {}).get("agent_action", {})
    if isinstance(action, dict):
        return action.get("message") or action.get("safe_next_step") or action.get("suggested_next_tool") or ""
    return ""


## Tool requests from an agent


In [ ]:
tool_requests = [
    {
        "request_id": "cortex-001",
        "tool_name": "run_revenue_sql",
        "sql": "SELECT region, account_status, SUM(arr) AS arr FROM ACMECLOUD_DEMO.PUBLIC.CUSTOMERS WHERE account_status = 'active' GROUP BY region, account_status",
        "operation": "read",
        "intended_use": "analytics",
        "actor_role": "DATA_ANALYST",
    },
    {
        "request_id": "cortex-002",
        "tool_name": "prepare_growth_campaign",
        "table_name": "ACMECLOUD_DEMO.PUBLIC.CUSTOMERS",
        "operation": "read",
        "intended_use": "marketing",
        "actor_role": "GROWTH_ANALYST",
        "columns": ["CUSTOMER_NAME", "EMAIL"],
    },
]


## Preflight wrapper


In [ ]:
def cortex_tool_preflight(request):
    if "sql" in request:
        response = client.validate_query_context(
            request["sql"],
            operation=request["operation"],
            intended_use=request["intended_use"],
            actor_role=request["actor_role"],
        )
    else:
        response = client.authorize_use(
            request["table_name"],
            operation=request["operation"],
            intended_use=request["intended_use"],
            actor_role=request["actor_role"],
            columns=request.get("columns"),
        )

    decision = decision_label(response)
    return {
        "request_id": request["request_id"],
        "tool_name": request["tool_name"],
        "decision": decision,
        "invoke_tool": decision != "DENY",
        "metatate_action": agent_action_text(response),
        "decision_id": response.get("data", {}).get("decision_id") or response.get("data", {}).get("validation_id"),
    }

preflight_results = pd.DataFrame([cortex_tool_preflight(request) for request in tool_requests])
preflight_results


In [ ]:
allowed = preflight_results[preflight_results["invoke_tool"]]
blocked = preflight_results[~preflight_results["invoke_tool"]]

print("Tools the agent may invoke:")
print(allowed[["request_id", "tool_name", "decision"]].to_string(index=False))
print("\nTools blocked before invocation:")
print(blocked[["request_id", "tool_name", "decision", "metatate_action"]].to_string(index=False))


This pattern keeps the agent runtime simple: every tool invocation gets a Metatate preflight decision, and denied tools never receive the data request.
